# Unified CAEN + Pico Scan Controller

This notebook replaces the manual workflow of juggling a Jupyter window + VS Code window.
All motor commands and data acquisition are driven from here.

**Layout assumption:**
```
your_project/
  scan_controller/
    scan_controller.py       ← unified controller
    pico/
      pico_controller.py     ← serial interface to Pico
  gui/
    wja_caen_tcal.py         ← your existing CAEN library
    nbutil.py
  notebooks/
    scan_notebook.ipynb      ← this file
```
Adjust `sys.path` in the next cell if your paths differ.

In [ ]:
%matplotlib inline
import sys, os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import time

plt.rcParams['figure.figsize'] = [12, 4]
matplotlib.rcParams['font.size'] = 14

# ── path setup ────────────────────────────────────────────────────────────────
# Point to the folder that contains scan_controller.py
SCAN_CTRL_DIR = os.path.abspath('../scan_controller')
if SCAN_CTRL_DIR not in sys.path:
    sys.path.insert(0, SCAN_CTRL_DIR)

# Point to the folder that contains wja_caen_tcal.py (your existing gui dir)
GUI_DIR = os.path.abspath('../gui')
if GUI_DIR not in sys.path:
    sys.path.insert(0, GUI_DIR)

from scan_controller import ScanController
print('Import OK')

## 1 – Connect

In [ ]:
# pico_port=None → auto-detect USB CDC port.  Override if needed:
# sc = ScanController(pico_port='/dev/ttyACM0')   # Linux
# sc = ScanController(pico_port='COM5')           # Windows

sc = ScanController(pico_port=None)
sc.connect(load_caen_corrections=True)
sc.status()

## 2 – Home axes

In [ ]:
sc.home_all()          # homes X first, then Y
# or individually:
# sc.pico.home('x')
# sc.pico.home('y')

## 3 – Quick test acquisition at current position

In [ ]:
sc.acquire(nevents=10)

# Plot a few channels
for ich in [0, 8, 16]:
    plt.figure()
    for ev in range(min(10, len(sc.caen.trigev))):
        plt.plot(sc.caen.trigev[ev].drsu[ich], alpha=0.5)
    plt.title(f'Channel {ich}')
    plt.xlabel('sample')
    plt.ylabel('V')
    plt.tight_layout()
    plt.show()

## 4 – Manual motor moves

In [ ]:
# Move to an absolute position (mm)
sc.move_to(x_mm=10.0, y_mm=5.0)
print(sc.position)

In [ ]:
# Relative move: +3 mm in X
sc.pico.move_mm('x', 3.0, forward=True)
print(sc.position)

## 5 – Line scan

In [ ]:
x_pts = np.arange(0, 30, 3.0)   # 0, 3, 6, … 27 mm

files = sc.line_scan(
    axis='x',
    positions=x_pts,
    fixed_other=5.0,       # Y held at 5 mm
    nevents=1000,
    output_dir='scan_data',
    label='linescan_y5mm',
    settle_time=0.5,
    home_first=True,
)
print('Files:', files)

## 6 – Grid scan (the main use case)

In [ ]:
x_positions = np.arange(0, 24, 3.0)   # 8 columns
y_positions = np.arange(0, 24, 3.0)   # 8 rows

files = sc.grid_scan(
    x_positions=x_positions,
    y_positions=y_positions,
    nevents=10000,
    output_dir='scan_data',
    label='grid_run1',
    settle_time=0.5,
    home_first=True,
    serpentine=True,        # minimise travel by reversing X on alternate rows
)
print(f'Grid scan complete: {len(files)} files')

## 7 – Arbitrary scan script (position list)

In [ ]:
# Fully custom point list — e.g. just a few specific positions
positions = [
    (0.0, 0.0),
    (5.0, 0.0),
    (5.0, 5.0),
    (0.0, 5.0),
]

files = sc.point_scan(
    positions=positions,
    nevents=500,
    output_dir='scan_data',
    label='custom_points',
    settle_time=0.3,
    home_first=False,      # already homed above
)
print('Files:', files)

## 8 – Disconnect

In [ ]:
sc.disconnect()